<h1 align="center">Visual Learning: linking transcriptomic identity to neural activity</h1>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h2>Overview</h2>

In the Visual Learning dataset, the same neurons are measured twice:

1. **In vivo**, with two-photon calcium imaging, while a mouse performs a visual change-detection task.
2. **Post hoc**, with spatial transcriptomics (HCR), which tells us which *cell type* each neuron is.

A coregistration pipeline links the two. That means we can ask a question you cannot ask
with either measurement alone: **does a neuron's transcriptomic cell type predict what it
does during behavior?**

Answering it requires two kinds of alignment, and most of this notebook is about getting them right:

- **Across modalities** — matching an imaged neuron to its transcriptomic cell type (Part 3)
- **Across sessions** — matching a neuron imaged on one day to the same neuron on another day (Part 7)

Everything here is done by **reading the NWB file directly** — no loader library, no wrapper class.
The point is that you can see every step: which container a piece of data lives in, what shape it
comes back as, and which identifier links it to the next piece.

Once the alignment is done, we make the same four plots for each session type:

| Plot | What it shows |
| --- | --- |
| Max projections by depth | where the neurons are, coloured by cell type |
| dF/F heatmaps | all activity in the session, then sorted by cell type |
| Tuning heatmap | which stimulus each neuron prefers |
| Change-aligned response | activity locked to the moment the image changes |

We do this first for a **gratings** session, then for two **natural image** sessions, and finally
compare the same neurons across two of those sessions.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h2>The data</h2>

Code Ocean mounts each attached data asset read-only under `/data`, in a folder named
after the asset. This tutorial needs three assets, plus the metadata table:

| Asset | What it provides |
| --- | --- |
| `Visual-Learning-SWDB` | one NWB per imaging session: activity, behavior, ROI masks |
| `Visual-Learning-Cell-Gene-Tables` | per-mouse cell x gene **AnnData**, carrying the cell-type labels |
| `cell-types-and-learning_coreg-id-mapping-assets-combined_2025-07-06` | the ID table linking imaged ROIs to HCR cells |
| `/data/metadata/visual_learning_session_metadata.csv` | one row per session: mouse, date, session type |

The coregistration asset still carries an older project name — this dataset used to be called
"cell types and learning" — so do not read anything into the mismatch. The code below finds it by
searching for the file it needs rather than by matching that name exactly.

The division of labour between the two HCR-side assets is worth being explicit about, because it is
the part people get wrong:

- The **coregistration asset** answers *"which HCR cell is this imaged ROI?"*. It is a table of
  **identifiers only** — no expression, no cell types.
- The **cell x gene AnnData** answers *"what type is that HCR cell, and what genes does it express?"*.
  It knows nothing about imaging.

Neither is useful alone. The coreg table's `hcr_id` is the key that opens the AnnData.

If a folder is missing, the asset is not attached — attach it in the capsule's Data panel.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h2>Setup</h2>

</div>

In [1]:
import ast
import os
import glob

import numpy as np
import pandas as pd
import anndata as ad
import pynwb
import matplotlib.pyplot as plt

# Show wide tables without pandas truncating the middle columns.
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)

# Every attached data asset appears as its own directory under /data.
data_dir = '/data'

plt.rcParams.update({
    'font.size': 14, 'axes.titlesize': 16, 'axes.labelsize': 15,
    'xtick.labelsize': 13, 'ytick.labelsize': 13, 'legend.fontsize': 13,
    'figure.titlesize': 18, 'figure.dpi': 100,
})

sorted(os.listdir(data_dir))

['409828_V1DD_Filtered',
 '416296_V1DD_Filtered',
 '427836_V1DD_Filtered',
 '438833_V1DD_Filtered',
 'Neuropixels_Opto_ecephys_nwb_combined',
 'Visual-Learning-SWDB',
 'brain-computer-interface-v2',
 'dynamicrouting_datacube',
 'metadata']

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h2>Part 1: Find sessions using the session metadata table</h2>

Before opening any data file, look at the **session metadata table**. It has one row per session
across the whole dataset — which mouse, which day, which training or imaging stage — so it is the
fastest way to find the sessions worth analyzing. Filenames alone will not tell you what stage a
session was.

</div>

In [2]:
# One row per imaging session, across every mouse in the dataset.
session_metadata = pd.read_csv(
    os.path.join(data_dir, 'metadata', 'visual_learning_session_metadata.csv'))

print('sessions:', session_metadata.shape)
print('mice:    ', sorted(session_metadata['subject_id'].unique()))
print()
print('columns:', session_metadata.columns.tolist())

sessions: (147, 25)
mice:     [np.int64(782149), np.int64(788406), np.int64(790322), np.int64(800792), np.int64(800995), np.int64(804363)]

columns: ['subject_id', 'session_id', 'name', 'session_type', 'acquisition_type', 'stage', 'image_set', 'session_number', 'acquisition_date', 'session_date', 'session_time', 'age_days', 'genotype', 'sex', 'date_of_birth', 'rig', 'project_name', 'n_planes', 'plane_names', 'imaging_depths', 'targeted_structures', 'processed_stamp', '_id', 'in_capsule', 'planes_failing_zdrift']


<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

This table is built from the **aind-data-schema v2** metadata, so a few column names differ from
what you may have seen in older notebooks. The ones we use:

| Column | Meaning |
| --- | --- |
| `subject_id` | which mouse |
| `session_id` | the acquisition name, which is how the NWB file is named |
| `name` | the full processed-asset name: `<session_id>_processed_<stamp>` |
| `session_number` | order the sessions were acquired in, per mouse |
| `session_date` | acquisition date — this becomes part of the ROI identifiers later on |
| `session_type` | the training or imaging stage (see the next section) |
| `stage` | just the stage prefix of `session_type`, e.g. `OPHYS_4` — handy for grouping |
| `image_set` | which set of natural images (A or B), if any |
| `n_planes`, `plane_names`, `imaging_depths` | the imaging planes and their depths in microns |
| `planes_failing_zdrift` | how many of this session's planes failed the z-drift QC check |

Three v2 details worth knowing before you index into it:

- **`acquisition_type` is the v2 name for `session_type`**, and both columns are present here
  holding identical values. Use either; this notebook uses `session_type`.
- **`plane_names`, `imaging_depths` and `targeted_structures` are lists stored as strings** —
  `"['VISp_0', 'VISp_1', ...]"` — because a CSV cell holds one value. Parse them with
  `ast.literal_eval` before using them, or you will be indexing into a string character by
  character.
- **`imaging_depths` is in the same order as `plane_names`, not sorted.** `VISp_0` is not
  necessarily the most superficial plane, so `zip` the two lists rather than assuming.

Pick one mouse and list its sessions in acquisition order. This is that animal's whole experimental
history, top to bottom.

</div>

In [3]:
# EDIT: any mouse in the table above. 800995 has all three session types this notebook uses.
mouse = 800995

mouse_sessions = (session_metadata[session_metadata['subject_id'] == mouse]
                  .sort_values('session_number'))

mouse_sessions[['session_number', 'session_date', 'session_type', 'stage', 'image_set',
                'age_days', 'n_planes', 'planes_failing_zdrift', 'session_id']]

,session_number,session_date,session_type,stage,image_set,age_days,n_planes,planes_failing_zdrift,session_id
105,1,2025-08-05,TRAINING_0_gratings_autorewards_15min,TRAINING_0,NaN,120,8,NaN,multiplane-ophys_800995_2025-08-05_13-25-58
106,2,2025-08-07,TRAINING_1_gratings,TRAINING_1,NaN,122,8,NaN,multiplane-ophys_800995_2025-08-07_12-29-04
107,3,2025-08-08,TRAINING_1_gratings,TRAINING_1,NaN,123,8,NaN,multiplane-ophys_800995_2025-08-08_12-49-42
108,4,2025-08-11,TRAINING_1_gratings,TRAINING_1,NaN,126,8,NaN,multiplane-ophys_800995_2025-08-11_11-02-36
109,5,2025-08-12,TRAINING_2_gratings_flashed,TRAINING_2,NaN,127,8,NaN,multiplane-ophys_800995_2025-08-12_10-33-26
110,6,2025-08-13,TRAINING_3_images_A_10uL_reward,TRAINING_3,A,128,8,0.0,multiplane-ophys_800995_2025-08-13_10-35-25
111,7,2025-08-14,TRAINING_3_images_A_10uL_reward,TRAINING_3,A,129,8,NaN,multiplane-ophys_800995_2025-08-14_12-22-10
112,8,2025-08-15,TRAINING_3_images_A_10uL_reward,TRAINING_3,A,130,8,NaN,multiplane-ophys_800995_2025-08-15_12-44-14
113,9,2025-08-18,TRAINING_4_images_A_training,TRAINING_4,A,133,8,NaN,multiplane-ophys_800995_2025-08-18_12-03-27
114,10,2025-08-19,TRAINING_5_images_A_epilogue,TRAINING_5,A,134,8,0.0,multiplane-ophys_800995_2025-08-19_10-54-26


<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>What the session types mean</h3>

Mice do not start out able to do this task. They learn it over weeks, through a sequence of
**training stages**, and only then are they imaged under the two-photon microscope. The
`session_type` string encodes where in that sequence a session sits, and it is the single most
important column in the table: two sessions can look identical in the file and mean completely
different things.

Reading a `session_type`, left to right:

- **prefix** — `TRAINING` (in the behavior facility), `OPHYS` (under the two-photon microscope),
  or `STAGE` (passive stimulus sessions with no task)
- **number** — position in the intended sequence
- **image set** — `images_A` or `images_B`, when natural images were used
- **suffix** — anything unique to that session, e.g. `_10uL_reward`, `_handoff_ready`

<h4>The task</h4>

In the change-detection task, images are flashed continuously and the mouse must **lick to report
when the image identity changes**. On a *go* trial the image changes: licking within the 750 ms
response window is a **hit** and earns water; not licking is a **miss**. On a *catch* trial a change
time is drawn but the image does not change: licking is a **false alarm**, withholding is a
**correct reject**. Licking before the scheduled change **aborts** the trial, which is what
discourages the mouse from licking indiscriminately.

<h4>Training stages</h4>

| `session_type` | What happens |
| --- | --- |
| `TRAINING_0_gratings_autorewards_15min` | First ever session. Static gratings change orientation and reward is delivered **automatically**, with no lick required — this builds the change-to-reward association and teaches the mouse to use the lick spout. |
| `TRAINING_1_gratings` | Same static gratings, but now reward is **contingent on licking** after the orientation change. |
| `TRAINING_2_gratings_flashed` | Gratings are now **flashed** (250 ms on, 500 ms gray) instead of held on screen. The mouse must compare what it sees now with what it saw before the gray screen, which adds a short-term memory component. |
| `TRAINING_3_images_A_10uL_reward` | Switch from gratings to **8 natural images** (set A), with a 10 uL reward volume. |
| `TRAINING_4_images_A_training` | Continued natural-image training. |
| `TRAINING_5_images_A_epilogue` / `_handoff_ready` / `_handoff_lapsed` | Final training stage. Once performance reaches criterion (d-prime > 1 on 2 of 3 consecutive days) the mouse is `handoff_ready` and waits for a microscope to free up; `handoff_lapsed` means performance dropped back below criterion while waiting; `epilogue` sessions append an extra passive stimulus block after the task. |

<h4>Imaging sessions</h4>

| `session_type` | What happens |
| --- | --- |
| `OPHYS_1_images_A` | Task performed under the microscope with the **familiar** image set A — the same images used throughout training. |
| `OPHYS_4_images_B` | **First exposure to novel image set B.** Same task, images the mouse has never seen. This is the novelty condition. |
| `OPHYS_6_images_B` | **Extinction.** *This differs from the Visual Behavior Ophys dataset, where OPHYS_6 is simply set B once it has become familiar.* Here the reward spout is still in place and the mouse can lick it, but **no rewards are delivered**. The mouse has to *unlearn* the stimulus-reward association it spent weeks acquiring. |
| `STAGE_0` | Passive natural movies, no task, no reward. |
| `STAGE_1` | Passive drifting gratings varying in contrast and temporal frequency, no task. |

Two details specific to the imaging sessions, both of which matter for analysis:

- **Omitted flashes.** During `OPHYS_*` sessions (but never during training) about 5% of non-change
  flashes are randomly **omitted**, leaving an extended gray period that breaks the expected
  rhythm of the stimulus. The change flash and the one before it are never omitted.
- **Gray-screen periods.** Each imaging session begins and ends with several minutes of gray
  screen, so spontaneous activity can be measured without stimulus or task.

For more on the task and the training curriculum, see the
[SWDB databook](https://allenswdb.github.io/physiology/stimuli/visual-behavior/VB-Behavior.html).

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Choose three sessions to compare</h3>

Rather than hardcoding dates, **select sessions by querying the table** — the same code then works
for any mouse. We take the *first* session of each type, because for the novel session "first" is
exactly what makes it novel.

The three we want:

- `TRAINING_1_gratings` — gratings have an orientation, so we can measure orientation tuning
- `OPHYS_1_images_A` — familiar natural images, imaged
- `OPHYS_4_images_B` — first exposure to novel images, imaged

One extra requirement, and it is easy to get caught by: **not every session has coregistration
data.** That pipeline runs per session and some sessions fail it, so a session can be in the
metadata table, have a perfectly good NWB, and still have no cell types attached. Since cell types
are the whole point here, we load the list of coregistered sessions now and require it — the
coregistration table itself is explained properly in Part 3.

</div>

In [ ]:
# Which sessions of this mouse have coregistration data? session_key is '<mouse>_<date>'.
coreg_path = glob.glob(
    os.path.join(data_dir, '**', f'{mouse}_coreg_id_mapping_table.csv'), recursive=True)[0]
coregistered_dates = set(pd.read_csv(coreg_path, usecols=['session_key'])['session_key']
                         .str.split('_').str[-1])

has_coreg = mouse_sessions['session_date'].isin(coregistered_dates)
print(f'{has_coreg.sum()} of {len(mouse_sessions)} sessions have coregistration data')
print('without:', mouse_sessions.loc[~has_coreg, 'session_date'].tolist())


def first_session_of_type(session_type):
    """Earliest session of this type for this mouse that also has coregistration data."""
    matching = mouse_sessions[(mouse_sessions['session_type'] == session_type) & has_coreg]
    if matching.empty:
        raise ValueError(f'mouse {mouse} has no coregistered {session_type} session')
    return matching.iloc[0]


# Each of these is a single row of the metadata table: a pandas Series.
gratings_session = first_session_of_type('TRAINING_1_gratings')
familiar_session = first_session_of_type('OPHYS_1_images_A')
novel_session = first_session_of_type('OPHYS_4_images_B')

for session in [gratings_session, familiar_session, novel_session]:
    # plane_names and imaging_depths are lists stored as strings -- parse before use, and
    # zip them together because imaging_depths follows plane order, not depth order.
    planes = ast.literal_eval(session['plane_names'])
    depths = ast.literal_eval(session['imaging_depths'])

    print(f"session {session['session_number']:>2}  {session['session_date']}  "
          f"{session['session_type']}")
    print('   planes:', dict(zip(planes, depths)))

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h2>Part 2: Open a session's NWB file and look inside</h2>

The NWB file is named after the `session_id` from the metadata table, so we can walk the asset
looking for that string rather than hardcoding a path.

One wrinkle: an NWB is either a single `.nwb` **file** (HDF5) or a `.nwb.zarr` **directory**. Zarr
stores an array as many small chunk files in a directory tree, which is what lets us read one piece
of a big dataset without loading all of it. `pynwb.read_nwb` opens either, but a directory walk has
to know not to descend into a `.nwb.zarr`, because from the outside it looks like an ordinary
folder.

</div>

In [ ]:
def find_nwb_paths(root, contains=None, max_depth=3):
    """Return paths to .nwb files and .nwb.zarr directories under `root`.

    contains : keep only paths containing this string -- pass the session id so a
               mount holding many sessions resolves to just yours.
    """
    found = []
    root = root.rstrip('/')
    base_depth = root.count(os.sep)

    for dirpath, dirnames, filenames in os.walk(root):
        # Stop descending past max_depth: data assets can be deep, and walking a
        # whole one is slow. Clearing dirnames in place is what prunes the walk.
        if dirpath.count(os.sep) - base_depth >= max_depth:
            dirnames[:] = []

        # A .nwb.zarr directory IS the file. Record it, and do NOT walk into its
        # internals, which hold thousands of chunk files.
        for d in list(dirnames):
            if d.endswith('.nwb.zarr'):
                found.append(os.path.join(dirpath, d))
                dirnames.remove(d)

        for f in filenames:
            if f.endswith('.nwb'):
                found.append(os.path.join(dirpath, f))

    if contains is not None:
        found = [p for p in found if contains in p]

    return sorted(found)          # sorted so the result is stable between runs


# The mount holding the ophys NWB files -- one of the names listed in Setup.
ophys_asset_dir = os.path.join(data_dir, 'Visual-Learning-SWDB')

gratings_nwb_matches = find_nwb_paths(ophys_asset_dir, contains=gratings_session['session_id'])

print(f"{len(gratings_nwb_matches)} NWB file(s) for {gratings_session['session_id']}")
for path in gratings_nwb_matches:
    print(' ', os.path.relpath(path, data_dir))

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

Now open it and **look at what is inside before indexing into it**. NWB sorts its contents into
a few top-level containers, and which one holds what varies between datasets, so print them all:

| container | commonly holds |
| --- | --- |
| `processing` | processed neural activity, and often behavior |
| `intervals` | trial tables, stimulus presentation tables, epoch tables |
| `acquisition` | raw acquired signals |
| `stimulus` | stimulus templates |

This session was imaged with a **mesoscope**, which records 8 planes at different depths
simultaneously. Each plane is a separate group under `processing`.

</div>

In [ ]:
gratings_nwb = pynwb.read_nwb(gratings_nwb_matches[0])

print('processing :', list(gratings_nwb.processing.keys()))
print('intervals  :', list(gratings_nwb.intervals.keys()) if gratings_nwb.intervals else [])
print('acquisition:', list(gratings_nwb.acquisition.keys()))
print('stimulus   :', list(gratings_nwb.stimulus.keys()) if gratings_nwb.stimulus else [])

In [ ]:
# Each imaging plane holds several versions of the activity, plus the segmentation.
print('inside one plane group:', list(gratings_nwb.processing['VISp_2'].data_interfaces.keys()))

# The ROI table sits under image_segmentation and has one row per segmented ROI.
gratings_example_roi_table = (gratings_nwb.processing['VISp_2']['image_segmentation']
                              .plane_segmentations['roi_table'].to_dataframe())

print('ROI table:', gratings_example_roi_table.shape)
print('columns  :', gratings_example_roi_table.columns.tolist())

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Reading one plane</h3>

`dff_timeseries` is the signal to use for most analyses. **&Delta;F/F** is the change in
fluorescence relative to each neuron's own baseline fluorescence; dividing by baseline makes
neurons comparable to each other, because a bright neuron and a dim one both read out as a
*fractional* change.

Three things to notice as we pull it out:

- **Shape.** The activity array is `(frames, ROIs)`.
- **Timestamps.** Some NWB timeseries store an explicit `timestamps` array, others store a sampling
  `rate` and you reconstruct the times. Check which you have — `series.timestamps` is `None` in the
  second case. Each mesoscope plane has *its own* timestamps, because the microscope visits the
  planes in sequence.
- **Lazy loading.** NWB data objects do not read from disk until you index them. That is what lets
  you open a huge file instantly, but it means you must slice with `[:]` to get a real array.

The imaging depth is available two ways, and we use the metadata table for it: the v2
`imaging_depths` column, paired with `plane_names`. The NWB carries the same number on the
**imaging plane** attached to the segmentation, inside a free-text `location` string
(`'Structure: VISp Depth: 160'`) — we print it below so you can see it, but a parsed column beats
a string you have to pull apart.

</div>

In [ ]:
plane = 'VISp_2'

# The dF/F trace for every ROI in this one plane.
dff_series = gratings_nwb.processing[plane]['dff_timeseries']['dff_timeseries']
plane_dff = dff_series.data[:]                    # (frames, ROIs) -- [:] actually reads it
plane_timestamps = dff_series.timestamps[:]       # seconds, on the same clock as behavior

# Depth per plane, from the v2 metadata columns. imaging_depths is in plane_names order.
gratings_depth_of_plane = dict(zip(ast.literal_eval(gratings_session['plane_names']),
                                   ast.literal_eval(gratings_session['imaging_depths'])))

# The same information as free text on the NWB's ImagingPlane, for comparison.
imaging_plane = (gratings_nwb.processing[plane]['image_segmentation']
                 .plane_segmentations['roi_table'].imaging_plane)

print(f'{plane}: dff {plane_dff.shape} (frames, ROIs) '
      f'at {gratings_depth_of_plane[plane]} um')
print(f'  NWB location string: {imaging_plane.location!r}')
print(f'  frame rate         : {1 / np.median(np.diff(plane_timestamps)):.2f} Hz')
print(f'  session duration   : {plane_timestamps[-1] / 60:.1f} min')

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Putting the 8 planes together, and naming every ROI</h3>

We want one activity matrix for the whole session: `(frames, neurons)` with all 8 planes side by
side, plus a table with one row per column of that matrix.

The important decision is in that table. Segmentation gives us an ROI table per plane whose
**row order is the only thing tying it to the activity array** — the `id` column in these files is
all zeros. Rather than carrying that fragile positional relationship around, we turn position into
a **name**, once, right here, in exactly the format the coregistration table uses:

| Column | Built from | Example |
| --- | --- | --- |
| `roi_id` | plane name + row index, zero-padded to 4 digits | `VISp_0_0010` |
| `unique_roi_id` | `<mouse>_<date>_` + `roi_id` | `800995_2025-08-21_VISp_0_0010` |

From here on every join is **on a string key**, not on an array offset. That is what makes it safe
to subset the data — filter to somas, filter to coregistered cells, drop a bad plane — in any
order, because each row carries its own identity with it. The positional assumption has to be got
right exactly once, and the `assert` below is what checks it.

Two more columns we keep for later:

- `dff_column` — which column of the concatenated activity matrix this ROI is
- `plane_roi_index` — which row of *this plane's* ROI table it came from, needed for the ROI masks

</div>

In [ ]:
# The imaging planes, from the metadata table rather than assumed to be VISp_0..VISp_7.
plane_names = ast.literal_eval(gratings_session['plane_names'])

# session_key is how the coregistration table names a session: <mouse>_<date>.
gratings_session_key = f"{mouse}_{gratings_session['session_date']}"

dff_per_plane = []
roi_rows_per_plane = []
gratings_timestamps = {}          # one timestamp array per plane

for plane in plane_names:
    dff_series = gratings_nwb.processing[plane]['dff_timeseries']['dff_timeseries']
    plane_dff = np.asarray(dff_series.data[:])
    gratings_timestamps[plane] = np.asarray(dff_series.timestamps[:])

    plane_segmentation = (gratings_nwb.processing[plane]['image_segmentation']
                          .plane_segmentations['roi_table'])
    roi_table = plane_segmentation.to_dataframe()

    # THE positional assumption, asserted once and then never relied on again:
    # row i of the ROI table is column i of this plane's dF/F matrix.
    assert len(roi_table) == plane_dff.shape[1], (
        f'{plane}: ROI table has {len(roi_table)} rows but dff has '
        f'{plane_dff.shape[1]} columns -- cannot assign ROI ids')

    roi_index = np.arange(len(roi_table))

    dff_per_plane.append(plane_dff)
    roi_rows_per_plane.append(pd.DataFrame({
        'plane': plane,
        'imaging_depth_um': gratings_depth_of_plane[plane],
        # position turned into a name, in the coregistration table's format
        'roi_id': [f'{plane}_{i:04d}' for i in roi_index],
        'unique_roi_id': [f'{gratings_session_key}_{plane}_{i:04d}' for i in roi_index],
        'plane_roi_index': roi_index,
        'is_soma': roi_table['is_soma'].values.astype(bool),
    }))

# Concatenate along the neuron axis: (frames, all ROIs in the session).
gratings_dff_all = np.concatenate(dff_per_plane, axis=1)

gratings_roi_table = pd.concat(roi_rows_per_plane, ignore_index=True)
gratings_roi_table['dff_column'] = np.arange(len(gratings_roi_table))

assert gratings_roi_table['unique_roi_id'].is_unique, 'unique_roi_id is not unique'

print('dff (frames, ROIs):', gratings_dff_all.shape)
print('soma ROIs: %d of %d' % (gratings_roi_table['is_soma'].sum(), len(gratings_roi_table)))
gratings_roi_table.head(3)

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>The behavior tables</h3>

Trials and stimulus presentations live in `intervals`, as NWB DynamicTables — call
`.to_dataframe()` and you have pandas. They are on the **same clock** as the imaging timestamps,
which is what makes alignment possible at all.

- `trials` — one row per behavioral trial, with `change_time` and what changed
- `stimulus_presentations` — one row per individual flash, including omitted ones

</div>

In [ ]:
gratings_trials = gratings_nwb.intervals['trials'].to_dataframe()
gratings_stimulus = gratings_nwb.intervals['stimulus_presentations'].to_dataframe()

print('trials              :', gratings_trials.shape)
print('stimulus_presentations:', gratings_stimulus.shape)
print()
print('trials columns:', gratings_trials.columns.tolist())

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h2>Part 3: Align ophys to transcriptomics</h2>

This is the first of the two alignments, and the reason the dataset exists. Everything we plot
afterwards depends on getting it right.

The link is made by a **coregistration table**, produced by matching the imaged neurons to cells in
the post-hoc HCR volume.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>The three ID systems</h3>

This is the part that trips people up. The imaging and the transcriptomics are two separate
measurements of the same tissue, and connecting them takes an intermediate step.

The physical chain is:

```
imaging plane  -->  structural stack  -->  HCR volume
  (one FOV,           (one z-stack of        (thin sections,
   one session)        the same volume)       gene expression)
```

Each imaging plane is registered into a **structural stack** — a high-resolution z-stack of the
same volume — and the structural stack is registered to the **HCR volume**. Going straight from a
2-photon plane to a tissue section is not tractable; the stack is the common reference frame that
makes both registrations possible. The stack is a step in the *pipeline*, not something you index
into here: it leaves its traces in the table's `cz_stack_id` / `resolved_cz_stack_id` / `max_iou`
columns, which are pipeline bookkeeping you can ignore for analysis.

The three identifiers you actually use:

| ID | Scope | What it identifies |
| --- | --- | --- |
| `unique_roi_id` | **one session, one plane** | one ROI mask, as segmented in this plane on this day |
| `unique_roicat_id` | **all sessions of one mouse** | a neuron, tracked across days |
| `hcr_id` | the mouse's HCR volume | the transcriptomic cell |

Why two IDs on the imaging side? Because a neuron imaged on Monday and again on Tuesday is the
*same cell* but a *different row* in each day's data. `unique_roi_id` is the row; `unique_roicat_id`
is the cell.

The coregistration table carries all three on the same row, which is what makes the link possible:

```
unique_roi_id  -->  unique_roicat_id  -->  hcr_id  -->  subclass
 (ROI mask, one       (the neuron,        (HCR cell)   (cell type)
  plane, one day)      across days)
   \_______ coreg id mapping table _______/   \__ HCR AnnData __/
```

Each end has a job: `unique_roi_id` finds the activity data, `unique_roicat_id` tracks a neuron
across sessions (Part 7), and `hcr_id` looks up the cell type.

</div>

In [ ]:
# coreg_path was located in Part 1; read the whole table now. One table per mouse,
# covering every session of that mouse the pipeline succeeded on.
print('reading', os.path.relpath(coreg_path, data_dir))

coreg_table = pd.read_csv(coreg_path, index_col=0)

print(coreg_table.shape,
      '|', coreg_table['session_key'].nunique(), 'sessions',
      '|', coreg_table['unique_roicat_id'].nunique(), 'distinct neurons')
print('matched: %d of %d rows' % (coreg_table['matched'].sum(), len(coreg_table)))
print()
print('columns:', coreg_table.columns.tolist())

coreg_table[['session_key', 'unique_roi_id', 'unique_roicat_id', 'plane_id',
             'matched', 'max_iou', 'cz_stack_id', 'hcr_id']].head(5)

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>What is in the coregistration table</h3>

One row per **coregistration-pipeline ROI per session**. Note what this is *not*: it is not every
ROI segmentation found. The pipeline only carries forward the ROIs it tracked across sessions, so
this table is a **subset** of each session's segmented ROIs — for the session we load below it is
about three quarters of them. Rows are present whether or not the match to a transcriptomic cell
ultimately succeeded.

| Column | Scope | Meaning |
| --- | --- | --- |
| `session_key` | — | `<mouse>_<date>`, which session this row belongs to |
| `session_name` | — | the full acquisition + processing folder name |
| `unique_roi_id` | one session | `<mouse>_<date>_<plane>_<index>`, one ROI mask on one day |
| `unique_roicat_id` | **all sessions** | the **neuron**, matched across sessions by ROICaT |
| `plane_id` | one session | which imaging plane, `VISp_0`-`VISp_7` |
| `matched` | — | whether this ROI was matched through the chain |
| `hcr_id` | HCR volume | the transcriptomic cell — **the key into the AnnData** |
| `cz_stack_id`, `resolved_cz_stack_id`, `max_iou`, `undecided`, `changed` | structural stack | pipeline bookkeeping from the stack-registration step; not needed for analysis |

Three things to check before joining, all of which will silently corrupt your results otherwise:

- **Unmatched entries are `-1`, not empty.** So `hcr_id == -1` means "no transcriptomic match", and
  if you join without filtering that first, every failed ROI joins to whatever cell sits at `-1`.
- **Not every session is in this table.** The coregistration pipeline runs per session and some
  sessions fail it, so a session that exists in the metadata table and has an NWB may still have
  zero coregistration rows. Check before you assume, which is why the cell below counts them.
- **Do not gate on the stack columns.** This is the filter that looks careful and quietly costs you
  data: many rows have `cz_stack_id == -1` and `max_iou == 0` and still carry a perfectly good
  `hcr_id` — in the session below, 45 of 243 HCR-matched ROIs are like that. Those columns record
  intermediate state from the stack-registration step, and `-1` there does not mean the
  transcriptomic match failed. **`hcr_id > 0` is the condition that decides whether an ROI has a
  cell type**, so filter on that and nothing else.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>The two ID scopes, and why both exist</h3>

This is the distinction to hold onto, because every join in the rest of the notebook depends on it.

- **`unique_roi_id` is one ROI mask, in one imaging plane, in one session.** It names a
  *segmentation result*: this blob of pixels, in this plane, on this day. Segmentation is run
  independently on each session, so a session has exactly one `unique_roi_id` per ROI it found, and
  the same neuron gets a *different* `unique_roi_id` in every session it appears in.
- **`unique_roicat_id` is one neuron, across all sessions of that mouse.** ROICaT matches ROI masks
  across sessions by their shape and position, and assigns each resulting neuron a single identifier
  that is stable across every session of that mouse. It is the unique identifier for a neuron — and
  a neuron detected in only one session still gets one, so having a `unique_roicat_id` does not by
  itself mean the cell was tracked across days.

So the relationship is **many `unique_roi_id` to one `unique_roicat_id`**, and how many varies per
neuron. A neuron detected on 12 of 20 days has 12 rows sharing one `unique_roicat_id`. This is why
Part 7 can track a neuron across days at all, and why the number of sessions a neuron appears in is
itself a data-quality variable.

Note also that `roi_id` is **not comparable between sessions**: `VISp_0_0010` on Monday and
`VISp_0_0010` on Tuesday are unrelated segmentations that happened to land at the same row index.
Only `unique_roicat_id` crosses sessions.

</div>

In [ ]:
# How many sessions does each neuron appear in?
sessions_per_neuron = (coreg_table.groupby('unique_roicat_id')['session_key'].nunique()
                       .value_counts().sort_index())

print('n_sessions -> n_neurons')
print(sessions_per_neuron.to_string())

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>The cell types come from the HCR AnnData</h3>

The cell-type labels live in the HCR cell-gene-tables asset, as an **AnnData** object
(`<mouse>_cellxgene_annotated.h5ad`) rather than a CSV. AnnData is the standard container for
single-cell data and it keeps expression and annotations together, which is why we prefer it here:

| Part | Shape | Contents |
| --- | --- | --- |
| `adata.X` | cells x genes | expression, raw spot counts per cell |
| `adata.layers['normalized']` | cells x genes | the same, normalized |
| `adata.obs` | cells x annotations | `class`, `subclass`, `cluster`, QC columns |
| `adata.var` | genes x annotations | `round`, `channel`, `gene` |

Crucially **`adata.obs_names` is the `cell_id`, and `cell_id` is the same identifier as the coreg
table's `hcr_id`.** That single fact is the whole join.

Note that `var_names` are *probe* names like `R5-514-Pvalb` — round, channel, gene — because a gene
is measured in a particular round and channel. The plain gene symbol is in `adata.var['gene']`.

</div>

In [ ]:
# The combined asset holds one folder per mouse, so search recursively for this
# mouse's cell x gene file rather than hardcoding the folder name.
hcr_matches = glob.glob(
    os.path.join(data_dir, 'Visual-Learning-Cell-Gene-Tables', '**',
                 f'{mouse}_cellxgene_annotated.h5ad'),
    recursive=True)

print(f'{len(hcr_matches)} match(es):', [os.path.relpath(p, data_dir) for p in hcr_matches])

adata = ad.read_h5ad(hcr_matches[0])

print(adata)
print('\nobs_names (= cell_id = hcr_id):', adata.obs_names[:5].tolist())

In [ ]:
# The three annotation columns we care about, coarse to fine.
adata.obs[['class', 'subclass', 'cluster']].value_counts().head(12)

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

Two conventions in `obs` to be aware of before joining:

- `subclass` is `'none'` for every cell that is not one of the four inhibitory subclasses this gene
  panel resolves — excitatory cells and unassigned cells alike. `'none'` is a **string, not a
  missing value**, so `dropna()` will not remove it. We convert it to `NaN` on the way in.
- `class` (`excitatory` / `inhibitory` / `unassigned`) is the coarser label, and it is the honest
  place to look at how many coregistered cells got a confident call at all.

We also have to fix a type mismatch: `hcr_id` is an integer in the coreg table and a string index
in the AnnData. Joins on mismatched types silently produce nothing.

</div>

In [ ]:
cell_types = adata.obs[['class', 'subclass', 'cluster']].copy()
cell_types.columns = ['cell_class', 'subclass', 'cluster_name']

# 'none' / 'unassigned' are placeholder strings, not labels -- make them real missing values.
cell_types['subclass'] = cell_types['subclass'].astype(str).replace('none', np.nan)
cell_types['cluster_name'] = cell_types['cluster_name'].astype(str).replace('unassigned', np.nan)

# Match the coreg table's integer hcr_id so the merge below actually finds anything.
cell_types.index = cell_types.index.astype(np.int64)
cell_types.index.name = 'hcr_id'

print(len(cell_types), 'HCR cells with annotations')
cell_types.head(3)

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Joining, then subsetting — in that order</h3>

Now put the two together. Because we gave every ROI a `unique_roi_id` in the coreg table's own
format back in Part 2, the ophys-to-transcriptomics link is a **string merge**, with no array
arithmetic anywhere:

```
roi_table.unique_roi_id  ==  coreg_table.unique_roi_id     (both '800995_2025-08-21_VISp_0_0010')
                             coreg_table.hcr_id  ==  adata.obs_names
```

We annotate first and subset second, deliberately. Annotating leaves the ROI table the same length
as the activity matrix, so nothing can fall out of step while we are still deciding what to keep.

</div>

In [ ]:
# Just this session's rows of the coregistration table.
gratings_coreg = coreg_table[coreg_table['session_key'] == gratings_session_key].copy()

# Unmatched entries are -1, not empty. Turn them into NaN BEFORE the join, or every
# failed ROI joins to whatever cell happens to sit at id -1.
gratings_coreg['hcr_id'] = gratings_coreg['hcr_id'].where(gratings_coreg['hcr_id'] > 0)

gratings_coreg = gratings_coreg[gratings_coreg['matched'].astype(bool)]

# HCR cell -> cell type. validate= makes pandas raise if the key is not unique on the right.
gratings_coreg = gratings_coreg.merge(cell_types, left_on='hcr_id', right_index=True,
                                     how='left', validate='many_to_one')

# hcr_id is the key into the AnnData and unique_roicat_id is the key across sessions;
# those two plus the labels are all the join needs.
coreg_columns = ['unique_roi_id', 'unique_roicat_id', 'hcr_id',
                 'cell_class', 'cluster_name', 'subclass']

# Merge on the ID string, NOT on (plane, position). ROIs with no coregistration get NaN;
# nothing is dropped here, so the table still lines up with the activity matrix.
gratings_roi_table = gratings_roi_table.merge(gratings_coreg[coreg_columns],
                                              on='unique_roi_id', how='left',
                                              validate='one_to_one')

print(f'{len(gratings_roi_table)} segmented ROIs')
print(f"  {gratings_roi_table['unique_roicat_id'].notna().sum():>5} in the coregistration table")
print(f"  {gratings_roi_table['hcr_id'].notna().sum():>5} matched to an HCR cell")
print(f"  {gratings_roi_table['subclass'].notna().sum():>5} with an inhibitory subclass label")

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>What that looks like across the whole dataset</h3>

One session is a small sample, so here is the same funnel measured over **77 sessions from three
mice** (782149, 788406, 790322).

**The two denominators answer different questions, so keep them straight.** A `unique_roi_id` is
one ROI mask in one plane in one session — a single segmentation result. A `unique_roicat_id` is a
*neuron*, and it gathers up every session in which that neuron was detected. Counting
`unique_roi_id`s weights a neuron by how many days it appeared, which is the right denominator for
"what fraction of my traces are somas" and the wrong one for "how many neurons do I have".

Per **`unique_roi_id`** (46,956 segmented ROI-sessions in total):

| Stage | ROI-sessions | `is_soma=True` | `is_soma=False` | % soma |
| --- | --- | --- | --- | --- |
| all segmented | 46,956 | 43,458 | 3,498 | 92.6 |
| in coreg table | 36,281 | 35,739 | 542 | 98.5 |
| `matched == True` | 35,085 | 34,583 | 502 | 98.6 |
| has `hcr_id` | 21,506 | 21,311 | 195 | 99.1 |

Read the other way: 82.2% of soma ROI masks reach the coreg table and 49.0% get an `hcr_id`, versus
15.5% and 5.6% of non-soma masks. **Coregistration selects for somas without being asked to** —
about ninefold. That follows from the method: the structural-stack match is a spatial-overlap test
against segmented cell bodies, so a dendrite has little to overlap with. `is_soma` was *not* used as
a coregistration criterion; the enrichment is a by-product, and it is a good reason to filter to
somas after coregistration.

Per **`unique_roicat_id`**, the same sessions hold 4,193 neurons, of which 1,525 have an `hcr_id`.
Because `is_soma` is assigned per session, a neuron can be called a soma on some days and not
others: **10.4% of HCR-matched neurons are mixed** that way. That is the number that bites in
cross-session work, because filtering per session drops those neurons from some sessions and keeps
them in others, silently shrinking any matched set and making it depend on which sessions you chose.
For multi-session analyses, decide `is_soma` **once per neuron** — a majority vote over its
sessions — and apply that decision uniformly.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Subsetting to the neurons we will analyze</h3>

Now, and only now, do we reduce the data. Two filters, both applied by **selecting rows of the ROI
table** and then slicing the activity matrix once to match:

1. `is_soma` — keep cell bodies, drop dendrites and segmentation artefacts
2. has an `hcr_id` — keep coregistered ROIs, since a cell type is the point of the notebook

After this cell, the ROI table and the activity matrix are the same length and in the same order,
and `dff_column` indexes the new array. We keep `plane_roi_index` because the ROI masks are read
from the NWB in full segmented order, so the mask plots still need it.

</div>

In [ ]:
keep_roi = gratings_roi_table['is_soma'] & gratings_roi_table['hcr_id'].notna()

# Slice the activity matrix ONCE, using the surviving rows' column indices.
gratings_dff = gratings_dff_all[:, gratings_roi_table.loc[keep_roi, 'dff_column'].values]

gratings_rois = gratings_roi_table[keep_roi].reset_index(drop=True)
gratings_rois['dff_column'] = np.arange(len(gratings_rois))    # index into the NEW matrix

print(f'{len(gratings_roi_table)} segmented ROIs -> {len(gratings_rois)} kept '
      f'(soma AND coregistered to an HCR cell)')
print('dff is now:', gratings_dff.shape)
print()
print(gratings_rois['cell_class'].value_counts(dropna=False).to_string())

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

Read those numbers in order, because each drop loses something different:

1. **Segmented -> soma + coregistered.** Most recorded ROIs never get an HCR match at all.
2. **Coregistered -> classified.** Of those that do, some come back `unassigned` — the
   transcriptomic call itself failed.
3. **Classified -> inhibitory subclass.** The gene panel is built to resolve *inhibitory* types.
   Excitatory cells are coregistered and classified, but have no subclass label here.

So the population in every plot below is a **thrice-filtered** subset. That is not a flaw to hide;
it is the sampling structure you have to reason about when interpreting any result.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Neurons with a subclass label</h3>

Three of the four plots group neurons by inhibitory subclass, so build that ordered subset once,
here, and give the four subclasses a fixed order and colour scheme (the standard Allen
inhibitory-subclass colours).

</div>

In [ ]:
# The inhibitory subclasses this HCR gene panel resolves, in standard order.
subclass_order = ['Pvalb', 'Sst', 'Vip', 'Lamp5']
subclass_colors = {'Pvalb': '#D93137', 'Sst': '#FF9900',
                   'Vip': '#A45FBF', 'Lamp5': '#DA808C'}

# Neurons with a subclass label, sorted so same-subclass neurons are adjacent rows in
# a heatmap. Categorical with an explicit order is what makes the sort respect our order
# rather than sorting alphabetically.
gratings_typed = gratings_rois.dropna(subset=['subclass']).copy()
gratings_typed['subclass'] = pd.Categorical(gratings_typed['subclass'],
                                            subclass_order, ordered=True)
gratings_typed = gratings_typed.sort_values(['subclass', 'cluster_name'])

print(f'{len(gratings_typed)} of {len(gratings_rois)} analyzed neurons have a subclass label')

# How are they distributed over imaging depth?
(gratings_typed.groupby(['imaging_depth_um', 'subclass'], observed=True)
 .size().unstack(fill_value=0))

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

Notice which depths have typed neurons and which do not.

The HCR method works on thick tissue but only penetrates the upper layers, so the deepest planes may
have few or no coregistered cells. **Typed neurons are not a random sample of recorded neurons** —
they are biased toward superficial depths, toward somas, and toward neurons that tracked well across
sessions. Any comparison between typed and untyped neurons has to account for that, for example by
matching on depth.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>The expression behind the labels</h3>

Because the labels came from an AnnData, the expression that produced them is right there. It is
worth one look: the marker genes should separate the subclasses, and if they do not, the labels are
not to be trusted.

We pull the coregistered cells out of the AnnData by `hcr_id` and average the normalized expression
of each subclass's canonical marker.

</div>

In [ ]:
marker_genes = ['Pvalb', 'Sst', 'Vip', 'Lamp5', 'Gad2', 'Slc17a7']

# var_names are probe names like 'R5-514-Pvalb'; the plain symbol is in var['gene'].
probe_of_gene = {gene: probe for probe, gene in zip(adata.var_names, adata.var['gene'])}

# obs_names are strings, hcr_id is a float column after the NaN conversion -- go via int.
recorded_hcr_ids = gratings_rois['hcr_id'].dropna().astype(np.int64).astype(str)
recorded_cells = adata[adata.obs_names.isin(recorded_hcr_ids)]

marker_expression = pd.DataFrame(
    recorded_cells[:, [probe_of_gene[g] for g in marker_genes]].layers['normalized'],
    index=recorded_cells.obs_names, columns=marker_genes)
marker_expression['subclass'] = recorded_cells.obs['subclass'].astype(str).values

(marker_expression.groupby('subclass')[marker_genes].mean()
 .reindex(subclass_order + ['none']).round(2))

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

Read this table **down the columns**: each subclass has the highest value in its own marker gene,
which is what makes the labels credible. `Gad2` (pan-inhibitory) is high in all four, and `Slc17a7`
(excitatory) is near zero everywhere — so no excitatory cells have leaked in.

The `none` row is the coregistered cells with no subclass call. Its `Slc17a7` is also low and its
`Gad2` is substantial, so these are mostly inhibitory cells the clustering could not confidently
place, not excitatory cells. That is consistent with the `cell_class` counts above, where the
unlabelled coregistered cells were `unassigned` rather than `excitatory`. Coregistration is biased
toward the sparse inhibitory population, because those are the cells this panel labels.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h2>Part 4: The four plots</h2>

Each plot is a small function so we can run the same set on any session, but each one takes the
pieces of data explicitly — the activity matrix, the ROI table, the timestamps — rather than a
session object. Nothing is hidden.

<h3>Plot 1: max projections by depth, ROIs coloured by cell type</h3>

The `max_projection` is the brightest value each pixel reached over the session, so active neurons
stand out against the background. ROI masks are stored as one image per ROI, of shape
**(ROIs, height, width)** — and critically, in the **full segmented order**, not our subset order,
because they come straight from the NWB.

This is the one place we need `plane_roi_index`: it says which row of the mask array each kept ROI
came from. The ROIs we filtered out are drawn as grey outlines, which is what shows the analyzed
neurons as a *fraction* of what was segmented.

</div>

In [ ]:
def plot_max_projections(nwb, typed_rois, depth_of_plane, title):
    """Max projection per plane, ordered by depth, with ROI masks drawn on top.

    depth_of_plane : {plane name: depth in microns}, from the session metadata row.
    """
    # (plane, plane_roi_index) -> subclass, for the typed neurons only.
    subclass_of_roi = typed_rois.set_index(['plane', 'plane_roi_index'])['subclass']

    # Order the panels from the brain surface downward, since imaging_depths follows
    # plane order rather than depth order.
    planes_by_depth = sorted(depth_of_plane, key=lambda p: depth_of_plane[p])

    fig, axes = plt.subplots(2, 4, figsize=(16, 9.5))

    for ax, plane in zip(axes.flat, planes_by_depth):
        segmentation = nwb.processing[plane]['image_segmentation'].plane_segmentations['roi_table']

        # The summary image and the masks, read on demand for this plane only.
        max_projection = np.asarray(nwb.processing[plane]['images']['max_projection'].data[:])
        roi_masks = np.asarray(segmentation['image_mask'].data[:])   # (all ROIs, h, w)

        # Percentile clipping: a few very bright pixels would otherwise wash the image out.
        low, high = np.percentile(max_projection, [1, 99.5])
        ax.imshow(max_projection, cmap='gray', vmin=low, vmax=high)

        n_typed = 0
        for roi_index in range(roi_masks.shape[0]):
            if (plane, roi_index) in subclass_of_roi.index:
                color = subclass_colors[subclass_of_roi[(plane, roi_index)]]
                ax.contourf(roi_masks[roi_index].astype(float), levels=[0.5, 1.5],
                            colors=[color], alpha=0.7)
                n_typed += 1
            else:
                ax.contour(roi_masks[roi_index], levels=[0.5], colors='lightgray',
                           linewidths=0.5)

        ax.set_title(f'{plane}, {depth_of_plane[plane]} ' + r'$\mu$m'
                     + f'\n{n_typed} of {roi_masks.shape[0]} typed', fontsize=14)
        ax.axis('off')

    legend_handles = [plt.Line2D([], [], marker='s', linestyle='', markersize=12,
                                 color=subclass_colors[name], label=name)
                      for name in subclass_order]
    fig.legend(handles=legend_handles, loc='lower center', ncol=4, frameon=False)

    fig.suptitle(title, y=0.99)
    fig.tight_layout(rect=[0, 0.04, 1, 0.97], h_pad=3)
    plt.show()


plot_max_projections(gratings_nwb, gratings_typed, gratings_depth_of_plane,
                     f"{gratings_session['session_date']}  --  {gratings_session['session_type']}")

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>A helper for the subclass groupings</h3>

Three of the four plots show neurons grouped by subclass, and each one needs to mark where the
groups start and end. Write that once.

The groups are marked two ways: a **colour strip** down the left edge, and a white divider line
between adjacent blocks. The strip is drawn in its own narrow axes so it does not eat into the
heatmap.

</div>

In [ ]:
def add_subclass_bar(ax, typed_rois, label=True):
    """Draw a subclass colour strip just left of a heatmap whose y axis is neurons.

    Assumes the heatmap rows are in the order of `typed_rois`, i.e. sorted by subclass.
    """
    block_sizes = typed_rois['subclass'].value_counts()[subclass_order].values
    block_ends = np.cumsum(block_sizes)
    block_starts = block_ends - block_sizes
    block_centres = block_ends - block_sizes / 2

    # A narrow inset axes glued to the left edge of ax. We set its limits to match rather
    # than using sharey, because sharing would make clearing ax's ticks clear the bar's too.
    bar = ax.inset_axes([-0.045, 0, 0.03, 1])
    bar.set_ylim(len(typed_rois), 0)          # inverted, to match imshow's row order
    bar.set_xlim(0, 1)
    bar.set_xticks([])

    for name, start, size in zip(subclass_order, block_starts, block_sizes):
        bar.axhspan(start, start + size, color=subclass_colors[name], linewidth=0)

    if label:
        bar.set_yticks(block_centres)
        bar.set_yticklabels(subclass_order)
        for tick, name in zip(bar.get_yticklabels(), subclass_order):
            tick.set_color(subclass_colors[name])
            tick.set_fontweight('bold')
        bar.tick_params(length=0, pad=4)
    else:
        bar.set_yticks([])

    for side in bar.spines.values():
        side.set_visible(False)

    # Dividers between adjacent blocks, drawn on the heatmap itself.
    for edge in block_ends[:-1]:
        ax.axhline(edge, color='white', linewidth=2)

    ax.set_yticks([])
    return bar

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Plot 2: dF/F heatmaps</h3>

One trace per neuron becomes unreadable past a handful of neurons, so use a **heatmap**: each row is
one neuron, x is time, colour is &Delta;F/F.

The activity matrix is `(frames, neurons)` and a heatmap wants `(neurons, frames)`, hence the `.T`.
Two panels: every analyzed neuron, then only those with a subclass label, grouped by subclass. The
second is a subset of the first, which is the point — it shows how much of the population carries a
cell-type label.

</div>

In [ ]:
def plot_dff_heatmaps(dff, typed_rois, timestamps, n_segmented, title):
    """Two heatmaps of the whole session: all analyzed neurons, then typed neurons."""
    time = timestamps[plane_names[0]]     # any plane's clock is fine for the x extent

    fig, axes = plt.subplots(2, 1, figsize=(14, 10))

    # extent puts real seconds on the x axis; the y extent counts neurons downward.
    axes[0].imshow(dff.T, aspect='auto', cmap='magma', vmin=0, vmax=2,
                   extent=[time[0], time[-1], dff.shape[1], 0])
    axes[0].set_ylabel('Neuron')
    axes[0].set_title(f'{dff.shape[1]} soma ROIs coregistered to an HCR cell '
                      f'(of {n_segmented} segmented)')

    # Reorder the columns to the subclass-sorted order before transposing.
    axes[1].imshow(dff[:, typed_rois['dff_column']].T, aspect='auto', cmap='magma',
                   vmin=0, vmax=2, extent=[time[0], time[-1], len(typed_rois), 0])
    add_subclass_bar(axes[1], typed_rois)
    axes[1].set_xlabel('Time (s)')
    axes[1].set_title(f'{len(typed_rois)} with an inhibitory subclass label')

    fig.suptitle(title)
    fig.tight_layout(rect=[0.055, 0, 1, 1])
    plt.show()


plot_dff_heatmaps(gratings_dff, gratings_typed, gratings_timestamps,
                  n_segmented=len(gratings_roi_table),
                  title=f"{gratings_session['session_date']}  --  {gratings_session['session_type']}")

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Building a change-aligned response</h3>

The two remaining plots both need the same thing: activity cut out around each image change.

The steps are: find which frame is closest to each time we want, pull out those frames, subtract
each trial's own pre-change baseline so every trial starts at zero, then average over trials.

Two subtleties, both of which change the answer:

- **Edge changes have to go.** A change too close to the start or end of the recording cannot fill a
  whole window. Drop those explicitly rather than letting the indexing silently clamp.
- **Rounding to the nearest frame, not the next one.** `np.searchsorted` gives the first frame at or
  *after* the time you ask for, so on average every sample comes from half a frame later than
  intended. At ~10 Hz that is ~50 ms of systematic shift — enough to make a response look like it
  begins slightly *before* the change. Checking the frame on either side and taking the closer one
  removes the bias.

</div>

In [ ]:
def nearest_frame(timestamps, wanted_times):
    """Index of the frame closest in time to each wanted time.

    np.searchsorted alone returns the first frame at or AFTER the wanted time, which
    biases every sample half a frame late -- enough to make responses look like they
    start before the change. Comparing against the frame before it removes the bias.
    """
    after = np.clip(np.searchsorted(timestamps, wanted_times), 1, len(timestamps) - 1)
    before = after - 1

    closer_after = (np.abs(timestamps[after] - wanted_times)
                    < np.abs(timestamps[before] - wanted_times))

    return np.where(closer_after, after, before)


def usable_change_times(trials, timestamps, window):
    """Image-change times that fall far enough inside every plane's recording.

    Every plane must cover the whole window, so we take the LATEST start and the
    EARLIEST stop across planes.
    """
    change_times = trials['change_time'].dropna().values

    latest_start = max(timestamps[plane][0] for plane in plane_names)
    earliest_stop = min(timestamps[plane][-1] for plane in plane_names)

    inside = ((change_times > latest_start - window[0])
              & (change_times < earliest_stop - window[-1]))

    return change_times[inside]


def align_to_changes(dff, rois, timestamps, change_times, window):
    """Baseline-subtracted response of every neuron around each image change.

    Returns (n_changes, n_window_samples, n_neurons). Planes are handled separately
    because each mesoscope plane has its own timestamps.
    """
    aligned = np.zeros((len(change_times), len(window), dff.shape[1]))

    for plane in plane_names:
        columns = rois.loc[rois['plane'] == plane, 'dff_column'].values
        if len(columns) == 0:
            continue

        # Broadcasting gives one frame index per (change, window sample).
        frames = nearest_frame(timestamps[plane],
                              change_times[:, np.newaxis] + window[np.newaxis, :])

        cut = dff[:, columns][frames]                  # (changes, window, neurons)

        # Each trial minus its OWN pre-change baseline, so slow drift across the
        # session does not end up in the response.
        aligned[:, :, columns] = cut - cut[:, window < 0, :].mean(axis=1, keepdims=True)

    return aligned

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

Two summaries of that array are used below, and they are averages over different axes:

- **`change_aligned`** — average over *changes*, giving `(window, neurons)`: the mean time course.
- **`response_per_change`** — average over the *first second after the change*, giving
  `(changes, neurons)`: one number per trial per neuron, which is what tuning and correlations need.

</div>

In [ ]:
# The peri-change window used for every alignment in this notebook: -2 s to +4 s at 100 ms.
window = np.arange(-2, 4, 0.1)

gratings_change_times = usable_change_times(gratings_trials, gratings_timestamps, window)

gratings_aligned = align_to_changes(gratings_dff, gratings_rois, gratings_timestamps,
                                    gratings_change_times, window)

# Average over changes -> the mean time course, (window, neurons).
gratings_change_aligned = gratings_aligned.mean(axis=0)

# Average over the first second after the change -> one number per change per neuron.
in_response_window = (window >= 0) & (window <= 1)
gratings_response_per_change = gratings_aligned[:, in_response_window, :].mean(axis=1)

print(f"{len(gratings_trials['change_time'].dropna())} changes in the trials table, "
      f'{len(gratings_change_times)} usable after dropping edge trials')
print('aligned            (changes, window, neurons):', gratings_aligned.shape)
print('change_aligned     (window, neurons)         :', gratings_change_aligned.shape)
print('response_per_change (changes, neurons)       :', gratings_response_per_change.shape)

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Plot 3: tuning heatmap</h3>

What "tuning" means depends on the stimulus. In gratings sessions the stimulus that changes has an
**orientation**; in natural image sessions it has an **image identity**. Both are columns of the
trials table, so we just read whichever one varies.

One trap here: in gratings sessions *both* columns are filled, but `change_image_name` holds strings
like `'gratings_0'`, `'gratings_180'`, which sort alphabetically into the wrong order.
`change_orientation` is numeric, so prefer it.

To compare neurons we **z-score** each one across conditions — subtract that neuron's mean and
divide by its standard deviation — so every row shows *relative* preference on the same scale.

</div>

In [ ]:
def change_stimulus_labels(trials, change_times):
    """What was shown at each usable change: orientation for gratings, image for images.

    Returns (column_name, one label per change). change_orientation is filled in on
    every trial including aborted ones, so we select the same trials we aligned to
    rather than calling dropna().
    """
    is_used_change = trials['change_time'].isin(change_times).values

    for column in ['change_orientation', 'change_image_name']:
        if column not in trials:
            continue
        values = trials[column].values[is_used_change]
        if len(pd.unique(values[pd.notna(values)])) > 1:
            return column, values

    raise ValueError('no varying stimulus column found in the trials table')


gratings_stimulus_column, gratings_change_labels = change_stimulus_labels(
    gratings_trials, gratings_change_times)

print('stimulus column:', gratings_stimulus_column)
print(pd.Series(gratings_change_labels).value_counts().sort_index().to_string())

In [ ]:
def plot_tuning(response_per_change, change_labels, typed_rois, stimulus_column, title):
    """Z-scored mean response per stimulus condition, all neurons then grouped by subclass."""
    conditions = sorted(pd.unique(change_labels))   # numeric for orientation, alphabetical for images

    # Mean response per condition -> (conditions, neurons), then z-score within each neuron.
    tuning = np.stack([response_per_change[change_labels == c].mean(axis=0) for c in conditions])
    tuning_z = (tuning - tuning.mean(axis=0)) / (tuning.std(axis=0) + 1e-9)

    fig, axes = plt.subplots(1, 2, figsize=(13, 8.5),
                             gridspec_kw={'wspace': 0.45, 'right': 0.86})

    # LEFT: every analyzed neuron, sorted by which condition it prefers.
    preference_order = np.argsort(tuning_z.argmax(axis=0))
    axes[0].imshow(tuning_z[:, preference_order].T, aspect='auto', cmap='RdBu_r',
                   vmin=-1.75, vmax=1.75)
    axes[0].set_ylabel('Neuron (sorted by preference)')
    axes[0].set_title(f'All {tuning_z.shape[1]} analyzed neurons')

    # RIGHT: only typed neurons, in subclass order.
    image = axes[1].imshow(tuning_z[:, typed_rois['dff_column']].T, aspect='auto',
                           cmap='RdBu_r', vmin=-1.75, vmax=1.75)
    add_subclass_bar(axes[1], typed_rois)
    axes[1].set_title('Sorted by cell type')

    for ax in axes:
        ax.set_xticks(range(len(conditions)))
        ax.set_xticklabels([str(c).replace('.0', '') for c in conditions], rotation=45)
        ax.set_xlabel('orientation (deg)' if 'orientation' in stimulus_column else 'image')

    colorbar_axes = fig.add_axes([0.90, 0.15, 0.02, 0.7])
    fig.colorbar(image, cax=colorbar_axes, label='z-scored response')
    fig.suptitle(title)
    plt.show()


plot_tuning(gratings_response_per_change, gratings_change_labels, gratings_typed,
            gratings_stimulus_column,
            f"{gratings_session['session_date']}  --  {gratings_session['session_type']}")

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

Look at the columns of the left panel. The **0&deg; and 180&deg;** columns resemble each other, and
so do 90&deg; and 270&deg;.

That is expected: a *static* grating at 0&deg; and one at 180&deg; are the same image. There are
really only two orientations here, each measured twice — which is a free reliability check, since a
genuinely tuned neuron should give the same answer both times.

Always look at the raw tuning matrix before computing a selectivity score from it.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Plot 4: change-aligned response by cell type</h3>

Two panels: a heatmap of every typed neuron's change-aligned trace, sorted by cell type, and the
mean trace per subclass with a shaded standard error.

The shading marks the stimulus. Natural image sessions flash the image for 250 ms every 750 ms, so
we shade the **changed** image blue and the **repeated** flashes grey. Static gratings in
`TRAINING_1` are held on screen for a couple of seconds, so there is a single blue span and no
repeats. Both cases are read from the stimulus table rather than hardcoded.

</div>

In [ ]:
def stimulus_spans(stimulus_presentations, window):
    """Where to shade: a list of (start, stop, is_change) in seconds relative to the change."""
    duration = np.median(stimulus_presentations['stop_time']
                         - stimulus_presentations['start_time'])
    interval = np.median(np.diff(stimulus_presentations['start_time']))

    # Is there room for another presentation inside the plotted window? If not, the
    # stimulus is held on screen and only the change itself shows.
    if interval > window[-1] - window[0]:
        return [(0, duration, True)]

    # Flashed: a regular train, one of which is the change at time 0.
    onsets = np.arange(window[0], window[-1], interval)
    onsets = onsets - onsets[np.argmin(abs(onsets))]      # put a flash exactly at 0

    return [(onset, onset + duration, abs(onset) < 0.01) for onset in onsets]


def shade_stimulus(ax, stimulus_presentations, window):
    """Shade the stimulus flashes on a change-aligned axis: change in blue, repeats grey."""
    for start, stop, is_change in stimulus_spans(stimulus_presentations, window):
        ax.axvspan(start, stop, color='#4C8FCC' if is_change else '#D9D9D9',
                   alpha=0.30, linewidth=0)
    ax.axvline(0, color='#2C5F8A', linestyle='--', linewidth=1)

In [ ]:
def plot_change_response(change_aligned, typed_rois, stimulus_presentations, window, title):
    """Change-aligned heatmap of typed neurons, plus the mean trace per subclass."""
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    image = axes[0].imshow(change_aligned[:, typed_rois['dff_column']].T, aspect='auto',
                           cmap='RdBu_r', vmin=-0.15, vmax=0.15,
                           extent=[window[0], window[-1], len(typed_rois), 0])
    add_subclass_bar(axes[0], typed_rois)
    axes[0].axvline(0, color='black', linestyle='--', linewidth=1)
    axes[0].set_xlabel('Time from change (s)')
    axes[0].set_title(f'{len(typed_rois)} typed neurons')
    fig.colorbar(image, ax=axes[0], label=r'$\Delta$F/F', fraction=0.04)

    shade_stimulus(axes[1], stimulus_presentations, window)

    for name in subclass_order:
        columns = typed_rois.loc[typed_rois['subclass'] == name, 'dff_column'].values
        if len(columns) == 0:
            continue
        mean_trace = change_aligned[:, columns].mean(axis=1)
        standard_error = change_aligned[:, columns].std(axis=1) / np.sqrt(len(columns))

        axes[1].plot(window, mean_trace, color=subclass_colors[name], linewidth=2,
                     label=f'{name} (n={len(columns)})')
        axes[1].fill_between(window, mean_trace - standard_error, mean_trace + standard_error,
                             color=subclass_colors[name], alpha=0.2)

    axes[1].axhline(0, color='gray', linewidth=0.5)
    axes[1].set_xlabel('Time from change (s)')
    axes[1].set_ylabel(r'$\Delta$F/F change from baseline')
    axes[1].set_title('Mean by subclass')
    axes[1].legend(frameon=False)

    fig.suptitle(title)
    fig.tight_layout(rect=[0.05, 0, 1, 1])
    plt.show()


plot_change_response(gratings_change_aligned, gratings_typed, gratings_stimulus, window,
                     f"{gratings_session['session_date']}  --  {gratings_session['session_type']}")

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

For the gratings session the response is a single bump that decays while the grating is still on
screen. Read the subclass means with the sample sizes in mind — some subclasses have only a dozen
neurons in one session.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h2>Part 5: A familiar natural image session</h2>

Now run the identical four plots on `OPHYS_1_images_A`, where the mouse performs the same task with
natural images it has seen for days.

What differs from gratings:

| | Gratings (`TRAINING_1`) | Natural images (`OPHYS_*`) |
| --- | --- | --- |
| What changes | orientation (4 values) | image identity (8 images) |
| Trials column | `change_orientation` | `change_image_name` |
| On screen | static, ~2.4 s, every ~10 s | flashed, 250 ms every 750 ms |
| Blank flashes | no | yes, ~5% of flashes **omitted** |

Because every function above reads the stimulus from the file, none of them need changing. What we
do have to repeat is the loading and joining — and repeating it once, explicitly, is the point: you
can see that the second session goes through exactly the same steps with nothing hidden.

</div>

In [ ]:
# --- open the NWB and read all 8 planes, exactly as in Part 2 ---
familiar_nwb = pynwb.read_nwb(
    find_nwb_paths(ophys_asset_dir, contains=familiar_session['session_id'])[0])

familiar_session_key = f"{mouse}_{familiar_session['session_date']}"

# Planes and depths for THIS session, from its metadata row.
familiar_depth_of_plane = dict(zip(ast.literal_eval(familiar_session['plane_names']),
                             ast.literal_eval(familiar_session['imaging_depths'])))
plane_names = list(familiar_depth_of_plane)

dff_per_plane = []
roi_rows_per_plane = []
familiar_timestamps = {}

for plane in plane_names:
    dff_series = familiar_nwb.processing[plane]['dff_timeseries']['dff_timeseries']
    plane_dff = np.asarray(dff_series.data[:])
    familiar_timestamps[plane] = np.asarray(dff_series.timestamps[:])

    plane_segmentation = (familiar_nwb.processing[plane]['image_segmentation']
                          .plane_segmentations['roi_table'])
    roi_table = plane_segmentation.to_dataframe()

    assert len(roi_table) == plane_dff.shape[1], f'{plane}: ROI table and dff disagree'
    roi_index = np.arange(len(roi_table))

    dff_per_plane.append(plane_dff)
    roi_rows_per_plane.append(pd.DataFrame({
        'plane': plane,
        'imaging_depth_um': familiar_depth_of_plane[plane],
        'roi_id': [f'{plane}_{i:04d}' for i in roi_index],
        'unique_roi_id': [f'{familiar_session_key}_{plane}_{i:04d}' for i in roi_index],
        'plane_roi_index': roi_index,
        'is_soma': roi_table['is_soma'].values.astype(bool),
    }))

familiar_dff_all = np.concatenate(dff_per_plane, axis=1)
familiar_roi_table = pd.concat(roi_rows_per_plane, ignore_index=True)
familiar_roi_table['dff_column'] = np.arange(len(familiar_roi_table))

familiar_trials = familiar_nwb.intervals['trials'].to_dataframe()
familiar_stimulus = familiar_nwb.intervals['stimulus_presentations'].to_dataframe()

print(familiar_session['session_date'], familiar_session['session_type'])
print('dff (frames, ROIs):', familiar_dff_all.shape)

In [ ]:
# --- join to coregistration + cell types, exactly as in Part 3 ---
familiar_coreg = coreg_table[coreg_table['session_key'] == familiar_session_key].copy()

familiar_coreg['hcr_id'] = familiar_coreg['hcr_id'].where(familiar_coreg['hcr_id'] > 0)

familiar_coreg = familiar_coreg[familiar_coreg['matched'].astype(bool)]
familiar_coreg = familiar_coreg.merge(cell_types, left_on='hcr_id', right_index=True,
                                      how='left', validate='many_to_one')

familiar_roi_table = familiar_roi_table.merge(familiar_coreg[coreg_columns],
                                              on='unique_roi_id', how='left',
                                              validate='one_to_one')

# --- subset to soma + coregistered, and slice the activity matrix once ---
keep_roi = familiar_roi_table['is_soma'] & familiar_roi_table['hcr_id'].notna()

familiar_dff = familiar_dff_all[:, familiar_roi_table.loc[keep_roi, 'dff_column'].values]
familiar_rois = familiar_roi_table[keep_roi].reset_index(drop=True)
familiar_rois['dff_column'] = np.arange(len(familiar_rois))

familiar_typed = familiar_rois.dropna(subset=['subclass']).copy()
familiar_typed['subclass'] = pd.Categorical(familiar_typed['subclass'],
                                            subclass_order, ordered=True)
familiar_typed = familiar_typed.sort_values(['subclass', 'cluster_name'])

print(f'{len(familiar_roi_table)} segmented -> {len(familiar_rois)} analyzed, '
      f'{len(familiar_typed)} with a subclass label')

In [ ]:
# --- align to image changes, exactly as in Part 4 ---
familiar_change_times = usable_change_times(familiar_trials, familiar_timestamps, window)

familiar_aligned = align_to_changes(familiar_dff, familiar_rois, familiar_timestamps,
                                   familiar_change_times, window)

familiar_change_aligned = familiar_aligned.mean(axis=0)

in_response_window = (window >= 0) & (window <= 1)
familiar_response_per_change = familiar_aligned[:, in_response_window, :].mean(axis=1)

familiar_stimulus_column, familiar_change_labels = change_stimulus_labels(
    familiar_trials, familiar_change_times)

print(f'{len(familiar_change_times)} usable changes | stimulus column: {familiar_stimulus_column}')
print()
print('the eight natural images of set A:')
print(pd.Series(familiar_change_labels).value_counts().sort_index().to_string())

In [ ]:
familiar_title = f"{familiar_session['session_date']}  --  {familiar_session['session_type']}"

plot_max_projections(familiar_nwb, familiar_typed, familiar_depth_of_plane, familiar_title)

In [ ]:
plot_dff_heatmaps(familiar_dff, familiar_typed, familiar_timestamps,
                  n_segmented=len(familiar_roi_table), title=familiar_title)

In [ ]:
plot_tuning(familiar_response_per_change, familiar_change_labels, familiar_typed,
            familiar_stimulus_column, familiar_title)

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Is that diagonal real?</h3>

The tuning heatmap has a striking diagonal — but be careful, because **sorting neurons by their
preferred condition produces a diagonal even in pure noise.** Each neuron's peak is put in its own
column by construction.

The honest check is **split-half reliability**: build the tuning curve twice from random halves of
the trials, and correlate the two. A neuron with a real preference gives the same answer both
times.

</div>

In [ ]:
def tuning_reliability(response_per_change, change_labels, seed=1):
    """Correlate tuning curves built from two random halves of the changes, per neuron."""
    conditions = sorted(pd.unique(change_labels))

    shuffled_trials = np.random.default_rng(seed).permutation(len(change_labels))
    half_a = shuffled_trials[:len(shuffled_trials) // 2]
    half_b = shuffled_trials[len(shuffled_trials) // 2:]

    # One tuning curve per half: (conditions, neurons)
    curve_a, curve_b = [
        np.stack([response_per_change[half][change_labels[half] == c].mean(axis=0)
                  for c in conditions])
        for half in (half_a, half_b)]

    return np.array([np.corrcoef(curve_a[:, i], curve_b[:, i])[0, 1]
                     for i in range(response_per_change.shape[1])])


familiar_reliability = tuning_reliability(familiar_response_per_change, familiar_change_labels)

print('median split-half r: %.2f' % np.nanmedian(familiar_reliability))
print('fraction of neurons with r > 0.5: %.2f' % np.nanmean(familiar_reliability > 0.5))

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

If the median is comfortably positive and a good fraction of neurons exceed 0.5, image preference
in this session is a real, repeatable property and the diagonal is not just the sorting.

Try running the same function on the gratings session to compare how reliable orientation tuning
is — there are only two distinct orientations there, so expect a different picture.

</div>

In [ ]:
plot_change_response(familiar_change_aligned, familiar_typed, familiar_stimulus, window,
                     familiar_title)

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

The traces **oscillate at 1.33 Hz**, which is the flash rate — the grey spans line up with each
subsequent peak, so the ringing is the stimulus, not noise. This is why we did gratings first: a
static stimulus gives one clean bump, which makes it much easier to recognise that the alignment is
correct before moving to a rhythmic stimulus.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h2>Part 6: A novel image session</h2>

`OPHYS_4_images_B` is the mouse's **first exposure to a novel image set**. Same four plots, same
code path — only the session row changes.

</div>

In [ ]:
# --- open and read all 8 planes ---
novel_nwb = pynwb.read_nwb(
    find_nwb_paths(ophys_asset_dir, contains=novel_session['session_id'])[0])

novel_session_key = f"{mouse}_{novel_session['session_date']}"

# Planes and depths for THIS session, from its metadata row.
novel_depth_of_plane = dict(zip(ast.literal_eval(novel_session['plane_names']),
                             ast.literal_eval(novel_session['imaging_depths'])))
plane_names = list(novel_depth_of_plane)

dff_per_plane = []
roi_rows_per_plane = []
novel_timestamps = {}

for plane in plane_names:
    dff_series = novel_nwb.processing[plane]['dff_timeseries']['dff_timeseries']
    plane_dff = np.asarray(dff_series.data[:])
    novel_timestamps[plane] = np.asarray(dff_series.timestamps[:])

    plane_segmentation = (novel_nwb.processing[plane]['image_segmentation']
                          .plane_segmentations['roi_table'])
    roi_table = plane_segmentation.to_dataframe()

    assert len(roi_table) == plane_dff.shape[1], f'{plane}: ROI table and dff disagree'
    roi_index = np.arange(len(roi_table))

    dff_per_plane.append(plane_dff)
    roi_rows_per_plane.append(pd.DataFrame({
        'plane': plane,
        'imaging_depth_um': novel_depth_of_plane[plane],
        'roi_id': [f'{plane}_{i:04d}' for i in roi_index],
        'unique_roi_id': [f'{novel_session_key}_{plane}_{i:04d}' for i in roi_index],
        'plane_roi_index': roi_index,
        'is_soma': roi_table['is_soma'].values.astype(bool),
    }))

novel_dff_all = np.concatenate(dff_per_plane, axis=1)
novel_roi_table = pd.concat(roi_rows_per_plane, ignore_index=True)
novel_roi_table['dff_column'] = np.arange(len(novel_roi_table))

novel_trials = novel_nwb.intervals['trials'].to_dataframe()
novel_stimulus = novel_nwb.intervals['stimulus_presentations'].to_dataframe()

# --- join to coregistration + cell types ---
novel_coreg = coreg_table[coreg_table['session_key'] == novel_session_key].copy()

novel_coreg['hcr_id'] = novel_coreg['hcr_id'].where(novel_coreg['hcr_id'] > 0)

novel_coreg = novel_coreg[novel_coreg['matched'].astype(bool)]
novel_coreg = novel_coreg.merge(cell_types, left_on='hcr_id', right_index=True,
                                how='left', validate='many_to_one')

novel_roi_table = novel_roi_table.merge(novel_coreg[coreg_columns], on='unique_roi_id',
                                        how='left', validate='one_to_one')

# --- subset, then align ---
keep_roi = novel_roi_table['is_soma'] & novel_roi_table['hcr_id'].notna()

novel_dff = novel_dff_all[:, novel_roi_table.loc[keep_roi, 'dff_column'].values]
novel_rois = novel_roi_table[keep_roi].reset_index(drop=True)
novel_rois['dff_column'] = np.arange(len(novel_rois))

novel_typed = novel_rois.dropna(subset=['subclass']).copy()
novel_typed['subclass'] = pd.Categorical(novel_typed['subclass'], subclass_order, ordered=True)
novel_typed = novel_typed.sort_values(['subclass', 'cluster_name'])

novel_change_times = usable_change_times(novel_trials, novel_timestamps, window)
novel_aligned = align_to_changes(novel_dff, novel_rois, novel_timestamps,
                                 novel_change_times, window)
novel_change_aligned = novel_aligned.mean(axis=0)

in_response_window = (window >= 0) & (window <= 1)
novel_response_per_change = novel_aligned[:, in_response_window, :].mean(axis=1)

novel_stimulus_column, novel_change_labels = change_stimulus_labels(
    novel_trials, novel_change_times)

print(novel_session['session_date'], novel_session['session_type'])
print(f'{len(novel_roi_table)} segmented -> {len(novel_rois)} analyzed, '
      f'{len(novel_typed)} with a subclass label')
print()
print('set B shares no images with set A:')
print(pd.Series(novel_change_labels).value_counts().sort_index().to_string())

In [ ]:
novel_title = f"{novel_session['session_date']}  --  {novel_session['session_type']}"

plot_max_projections(novel_nwb, novel_typed, novel_depth_of_plane, novel_title)

In [ ]:
plot_dff_heatmaps(novel_dff, novel_typed, novel_timestamps,
                  n_segmented=len(novel_roi_table), title=novel_title)

In [ ]:
plot_tuning(novel_response_per_change, novel_change_labels, novel_typed,
            novel_stimulus_column, novel_title)

In [ ]:
plot_change_response(novel_change_aligned, novel_typed, novel_stimulus, window, novel_title)

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h2>Part 7: Align across sessions</h2>

The second alignment. We have the same four plots for a familiar and a novel session, but so far each
was analyzed on its own. To compare them we need to know **which neurons are the same**.

The image sets are disjoint — set A is `im061`-`im085`, set B is `im000`-`im106` — so we cannot ask
how the response to a particular image changed. We can only compare each *neuron* to itself.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Matching neurons across sessions</h3>

This is what `unique_roicat_id` is for. Recall the two scopes:

- `unique_roi_id` names **one ROI mask, in one plane, in one session**
- `unique_roicat_id` names **the neuron**, and is stable across all sessions of this mouse

Note that this alignment does not involve the HCR data at all: the cross-session tracking is done by
ROICaT on the imaging data alone. The cell type comes along for the ride, because the same
coregistration row carries both identifiers.

Both sessions have already been subset to soma + coregistered ROIs, so every row has a
`unique_roicat_id`. Take the neurons present in both by intersecting the two sets.

One caveat carried over from Part 3: because we applied the soma filter **per session**, a neuron
classified as a soma on one day but not the other drops out of this intersection. For a single pair
of sessions that is a small effect, but across many sessions it compounds — which is when deciding
`is_soma` once per neuron starts to matter.

</div>

In [ ]:
neurons_in_familiar = set(familiar_rois['unique_roicat_id'].dropna())
neurons_in_novel = set(novel_rois['unique_roicat_id'].dropna())

matched_neuron_ids = sorted(neurons_in_familiar & neurons_in_novel)

print(len(neurons_in_familiar), 'coregistered neurons in the familiar session')
print(len(neurons_in_novel), 'coregistered neurons in the novel session')
print(len(matched_neuron_ids), 'in both')

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

Now build the lookup: for each matched neuron, which **column of the activity matrix** does it
occupy in each session?

Setting `unique_roicat_id` as the index and then selecting `matched_neuron_ids` guarantees both
tables come out in the *same order*, so row *i* of one is the same neuron as row *i* of the other.
Getting this wrong is the easiest way to produce a completely meaningless result that looks fine.

</div>

In [ ]:
familiar_matched = familiar_rois.set_index('unique_roicat_id').loc[matched_neuron_ids]
novel_matched = novel_rois.set_index('unique_roicat_id').loc[matched_neuron_ids]

matched_neurons = pd.DataFrame({
    'unique_roicat_id': matched_neuron_ids,
    'familiar_column': familiar_matched['dff_column'].values,
    'novel_column': novel_matched['dff_column'].values,
    'familiar_plane': familiar_matched['plane'].values,
    'novel_plane': novel_matched['plane'].values,
    'hcr_id': familiar_matched['hcr_id'].values,
    'subclass': familiar_matched['subclass'].values,
})

# The same physical neuron must map to the same HCR cell in both sessions -- check, don't assume.
same_hcr_id = (familiar_matched['hcr_id'].values == novel_matched['hcr_id'].values)
print(f'{same_hcr_id.sum()} of {len(matched_neurons)} matched neurons agree on hcr_id '
      f'across sessions')

matched_neurons.head()

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

The `hcr_id` agreement is a genuine consistency check on the coregistration, and it should be 100%:
`unique_roicat_id` and `hcr_id` are both properties of the *neuron*, not of the session, so two rows
describing the same neuron must carry the same HCR cell. Any disagreement means the coregistration
is inconsistent between those two sessions, and those neurons should be dropped.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Two checks worth running</h3>

First: a matched neuron should be in the **same imaging plane** in both sessions. The mesoscope
targets the same depths each day, so a neuron jumping planes would mean the match is wrong.

Second: matched neurons should almost never share a **column index**, and that is the point of the
whole ID system.

</div>

In [ ]:
same_plane = matched_neurons['familiar_plane'] == matched_neurons['novel_plane']
same_column = matched_neurons['familiar_column'] == matched_neurons['novel_column']

print(f'{same_plane.sum()} of {len(matched_neurons)} matched neurons are in the same plane')
print(f'{same_column.sum()} of {len(matched_neurons)} matched neurons have the same column index')

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

Almost none of them share a column index, and that is exactly why `unique_roicat_id` exists.
Segmentation runs independently on each session, so it finds a slightly different set of ROI masks
in a different order every day. Column 17 in the familiar session is **not** the same neuron as
column 17 in the novel session.

If you ever compare two sessions by row position, this is the number that tells you the result is
meaningless.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Does the same neuron respond consistently?</h3>

Before asking how novelty changed anything, check that a neuron's response is a stable property of
that neuron at all. We already have one number per neuron per change in `response_per_change` — the
mean response in the first second after each change — so average over changes and correlate across
the matched pairs.

The shuffled control is what makes this interpretable: if the pairing were arbitrary, the
correlation should collapse.

</div>

In [ ]:
familiar_response = familiar_response_per_change.mean(axis=0)[matched_neurons['familiar_column']]
novel_response = novel_response_per_change.mean(axis=0)[matched_neurons['novel_column']]

r_matched = np.corrcoef(familiar_response, novel_response)[0, 1]

# What would we get if the pairing were wrong? Shuffle one side.
shuffled_novel_response = np.random.default_rng(0).permutation(novel_response)
r_shuffled = np.corrcoef(familiar_response, shuffled_novel_response)[0, 1]

print(f'correctly paired: r = {r_matched:.2f}')
print(f'shuffled pairing: r = {r_shuffled:.2f}')

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6.5))

# Untyped neurons in grey behind, so the typed ones are readable on top.
is_untyped = matched_neurons['subclass'].isna().values
ax.scatter(familiar_response[is_untyped], novel_response[is_untyped],
           color='lightgray', s=18, label=f'no cell type (n={is_untyped.sum()})')

for name in subclass_order:
    is_subclass = (matched_neurons['subclass'] == name).values
    ax.scatter(familiar_response[is_subclass], novel_response[is_subclass],
               color=subclass_colors[name], s=45, label=f'{name} (n={is_subclass.sum()})')

# Square axes covering ALL the data, with a small margin. Derive the limits rather than
# hardcoding them: a fixed range silently clips whichever neurons fall outside it, and a
# clipped scatter looks exactly like a complete one.
low = min(familiar_response.min(), novel_response.min())
high = max(familiar_response.max(), novel_response.max())
margin = 0.05 * (high - low)
axis_limits = [low - margin, high + margin]

# Unity line: points above it responded more to novel images, below more to familiar.
ax.plot(axis_limits, axis_limits, color='gray', linestyle='--', linewidth=1)

ax.set_xlim(axis_limits)
ax.set_ylim(axis_limits)
ax.set_aspect('equal')
ax.set_xlabel('Familiar images (OPHYS_1)')
ax.set_ylabel('Novel images (OPHYS_4)')
ax.set_title(f'Change response of {len(matched_neurons)} matched neurons\nr = {r_matched:.2f}')
ax.legend(frameon=False, fontsize=11)
plt.show()

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

A high correlation with a near-zero shuffled control tells us two things:

1. The matching is real. Random pairings give nothing.
2. How strongly a neuron responds to a change is a **stable property of that neuron**, holding up
   across a several-day gap and an entirely different image set.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Familiar versus novel, by cell type</h3>

Now the two sessions on the same axes, using **only the matched neurons**, so any difference cannot
be caused by a different set of cells. Solid is familiar, dashed is novel.

</div>

In [ ]:
fig, axes = plt.subplots(1, len(subclass_order), figsize=(17, 4.5), sharey=True)

for ax, name in zip(axes, subclass_order):
    is_subclass = (matched_neurons['subclass'] == name).values

    ax.plot(window,
            familiar_change_aligned[:, matched_neurons['familiar_column'][is_subclass]].mean(axis=1),
            color=subclass_colors[name], linewidth=2, label='familiar (A)')
    ax.plot(window,
            novel_change_aligned[:, matched_neurons['novel_column'][is_subclass]].mean(axis=1),
            color=subclass_colors[name], linestyle='--', linewidth=2, label='novel (B)')

    shade_stimulus(ax, familiar_stimulus, window)
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.set_title(f'{name} (n={is_subclass.sum()})')
    ax.set_xlabel('Time from change (s)')
    ax.legend(frameon=False, fontsize=11)

axes[0].set_ylabel(r'$\Delta$F/F change')
fig.tight_layout()
plt.show()

In [ ]:
matched_summary = pd.DataFrame({'familiar': familiar_response,
                                'novel': novel_response,
                                'subclass': matched_neurons['subclass'].values})

(matched_summary.groupby('subclass', observed=True)[['familiar', 'novel']]
 .agg(['count', 'mean']).round(4))

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

Read this table with the sample sizes in mind. Some subclasses have only a dozen matched neurons in
one pair of sessions, which is not enough to conclude anything about that cell type. The way to make
such a comparison convincing is to repeat it over many session pairs and many mice, which is what
the full dataset supports.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>One neuron, two sessions, one transcriptome</h3>

The plot that shows what all this ID-matching bought us: the same physical neuron, recorded on two
days under two different image sets, with its cell type and its measured gene expression — three
separate measurements resolved to one cell.

</div>

In [ ]:
# Pick the most strongly responding typed neuron, so the traces are legible.
has_subclass = matched_neurons['subclass'].notna().values
example_index = np.where(has_subclass)[0][np.argmax(familiar_response[has_subclass])]
example_neuron = matched_neurons.iloc[example_index]

print(example_neuron['unique_roicat_id'], '|', example_neuron['subclass'],
      '| hcr_id', int(example_neuron['hcr_id']))
print('familiar: plane', example_neuron['familiar_plane'],
      'column', example_neuron['familiar_column'])
print('novel:    plane', example_neuron['novel_plane'],
      'column', example_neuron['novel_column'])

# Its top-expressed genes, straight out of the AnnData -- indexed by hcr_id.
example_cell = adata[str(int(example_neuron['hcr_id']))]
example_expression = pd.Series(np.asarray(example_cell.layers['normalized']).ravel(),
                               index=adata.var['gene'].values)

print('\ntop genes:')
print(example_expression.sort_values(ascending=False).head(6).round(2).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

for dff, rois, timestamps, change_aligned, column, label, linestyle in [
        (familiar_dff, familiar_rois, familiar_timestamps, familiar_change_aligned,
         example_neuron['familiar_column'], f"familiar (A) -- {familiar_session['session_date']}", '-'),
        (novel_dff, novel_rois, novel_timestamps, novel_change_aligned,
         example_neuron['novel_column'], f"novel (B) -- {novel_session['session_date']}", '--')]:

    # This neuron's own plane, so we use the right clock for the whole-session trace.
    plane = rois['plane'][column]
    axes[0].plot(timestamps[plane], dff[:, column], linewidth=0.4, label=label)

    axes[1].plot(window, change_aligned[:, column], color='black',
                 linestyle=linestyle, linewidth=2, label=label)

axes[0].set_xlabel('Time from session start (s)')
axes[0].set_ylabel(r'$\Delta$F/F')
axes[0].set_title('Whole session')
axes[0].legend(frameon=False, fontsize=11)

shade_stimulus(axes[1], familiar_stimulus, window)
axes[1].set_xlabel('Time from change (s)')
axes[1].set_title('Change-aligned')
axes[1].legend(frameon=False, fontsize=11)

fig.suptitle(f"One {example_neuron['subclass']} neuron "
             f"(hcr_id {int(example_neuron['hcr_id'])}), two sessions")
fig.tight_layout()
plt.show()

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h2>Where to go next</h2>

The pieces are now in place to ask the questions this dataset was built for.

- **Follow neurons through the whole curriculum.** The `unique_roicat_id` intersection generalizes to
  any number of sessions. Intersect across all of them to get the neurons tracked from naive to
  expert, and watch their responses change. Decide `is_soma` once per neuron rather than per session
  before you do.
- **The novelty effect, properly.** Compare `OPHYS_1_images_A` (familiar), `OPHYS_4_images_B`
  (novel), and `OPHYS_6_images_B` (extinction: set B, no rewards). The three-way comparison separates
  novelty from image set from reward.
- **Extinction as its own question.** In `OPHYS_6_images_B` the mouse keeps licking at first and
  gets nothing, so the stimulus-reward association is being unlearned across the session. Splitting
  that session into early and late blocks, or tracking the lick rate alongside the neural response,
  is a learning question you can ask within a single session.
- **Change what you keep.** Drop the `is_soma` or the `hcr_id` filter in the subsetting cells and
  re-run any plot on a different population — for instance to check that a result on coregistered
  cells also holds in the full recorded population.
- **Use expression as a continuous variable.** Everything above treated cell type as a discrete
  label, but `adata.layers['normalized']` gives graded expression per cell. Correlating a functional
  metric against a single gene avoids committing to a clustering at all.
- **Use the finer clusters.** `adata.obs['cluster']` has ~30 clusters where `subclass` has four.
  There are fewer coregistered neurons per cluster, so this needs pooling across mice.
- **Omitted flashes.** Align to `omitted == True` in the stimulus table instead of to changes.
- **Hits versus misses.** Split `change_time` by trial outcome. Interpret differences beyond about a
  second after the change with care: by then the mouse has licked and consumed reward, so movement
  and reward are mixed in with vision.
- **Other mice.** `session_metadata['subject_id'].unique()` lists them. Each has its own
  coregistration table in the same asset and its own HCR AnnData.

Three cautions carried over from earlier parts. First, coregistered neurons are not a random sample
of segmented ROIs — they are biased toward superficial depths, toward somas, and toward neurons that
tracked reliably, so compare like with like. Second, all joins go through `unique_roi_id` and
`unique_roicat_id`; the moment you find yourself indexing one session's array with another session's
positions, stop. Third, a selectivity index built as a ratio saturates when one of the two responses
is negative; check the sign before trusting the value.

</div>